In [14]:
import pandas as pd

df = pd.read_csv(
    "/content/clin_trials.csv",
    engine="python",
    on_bad_lines="skip"
)

df.columns = df.columns.str.strip()

print(df.shape)
print(df.columns.tolist())

(496615, 17)
['Unnamed: 0', 'Organization Full Name', 'Organization Class', 'Responsible Party', 'Brief Title', 'Full Title', 'Overall Status', 'Start Date', 'Standard Age', 'Conditions', 'Primary Purpose', 'Interventions', 'Intervention Description', 'Study Type', 'Phases', 'Outcome Measure', 'Medical Subject Headings']


In [15]:
# Standardize text fields
text_columns = [
    "Organization Full Name",
    "Organization Class",
    "Responsible Party",
    "Brief Title",
    "Full Title",
    "Overall Status",
    "Standard Age",
    "Conditions",
    "Primary Purpose",
    "Interventions",
    "Intervention Description",
    "Study Type",
    "Phases",
    "Outcome Measure",
    "Medical Subject Headings"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

In [16]:
# Standardize key categorical fields
if "Overall Status" in df.columns:
    df["Overall Status"] = df["Overall Status"].str.upper()

if "Phases" in df.columns:
    df["Phases"] = df["Phases"].str.upper().replace({"": "UNKNOWN"})

if "Study Type" in df.columns:
    df["Study Type"] = df["Study Type"].str.title()

if "Primary Purpose" in df.columns:
    df["Primary Purpose"] = df["Primary Purpose"].str.title()

In [17]:
# Add IDs and derived fields
df = df.reset_index(drop=True)
df["trial_id"] = df.index + 1
df["success"] = df["Overall Status"].apply(lambda x: 1 if x == "COMPLETED" else 0)

df["start_date_parsed"] = pd.to_datetime(df["Start Date"], errors="coerce")
df["start_year"] = df["start_date_parsed"].dt.year

In [18]:
# 1. clinical_trials table
clinical_trials = df[[
    "trial_id",
    "Brief Title",
    "Full Title",
    "Overall Status",
    "Study Type",
    "Phases",
    "Primary Purpose",
    "Start Date",
    "Outcome Measure",
    "success"
]].copy()

clinical_trials.columns = [
    "trial_id",
    "brief_title",
    "full_title",
    "overall_status",
    "study_type",
    "phases",
    "primary_purpose",
    "start_date",
    "outcome_measure",
    "success"
]

In [20]:
# 2. organizations table
organizations = df[[
    "Organization Full Name",
    "Organization Class",
    "Responsible Party"
]].drop_duplicates().reset_index(drop=True).copy()

organizations["organization_id"] = organizations.index + 1

organizations = organizations[[
    "organization_id",
    "Organization Full Name",
    "Organization Class",
    "Responsible Party"
]]

organizations.columns = [
    "organization_id",
    "organization_full_name",
    "organization_class",
    "responsible_party"
]

In [21]:
# 3. conditions table
condition_rows = []

for _, row in df.iterrows():
    raw_conditions = row.get("Conditions", "")
    if raw_conditions:
        for cond in str(raw_conditions).split(","):
            cond = cond.strip()
            if cond:
                condition_rows.append({
                    "trial_id": row["trial_id"],
                    "condition_name": cond
                })

conditions = pd.DataFrame(condition_rows).drop_duplicates().reset_index(drop=True)
conditions["condition_id"] = conditions.index + 1
conditions = conditions[["condition_id", "trial_id", "condition_name"]]

In [22]:
# 4. interventions table
intervention_rows = []

for _, row in df.iterrows():
    raw_interventions = row.get("Interventions", "")
    description = row.get("Intervention Description", "")
    if raw_interventions:
        for inter in str(raw_interventions).split(","):
            inter = inter.strip()
            if inter:
                intervention_rows.append({
                    "trial_id": row["trial_id"],
                    "intervention_name": inter,
                    "intervention_description": description
                })

interventions = pd.DataFrame(intervention_rows).drop_duplicates().reset_index(drop=True)
interventions["intervention_id"] = interventions.index + 1
interventions = interventions[["intervention_id", "trial_id", "intervention_name", "intervention_description"]]

In [23]:
# Save cleaned outputs
output_dir = "/content/cleaned_outputs"
os.makedirs(output_dir, exist_ok=True)

cleaned_main = os.path.join(output_dir, "clinical_trials_cleaned.csv")
trials_path = os.path.join(output_dir, "clinical_trials.csv")
orgs_path = os.path.join(output_dir, "organizations.csv")
conds_path = os.path.join(output_dir, "conditions.csv")
inters_path = os.path.join(output_dir, "interventions.csv")

df.to_csv(cleaned_main, index=False)
clinical_trials.to_csv(trials_path, index=False)
organizations.to_csv(orgs_path, index=False)
conditions.to_csv(conds_path, index=False)
interventions.to_csv(inters_path, index=False)

print("Saved files:")
for p in [cleaned_main, trials_path, orgs_path, conds_path, inters_path]:
    print("-", p)

print("clinical_trials shape:", clinical_trials.shape)
print("organizations shape:", organizations.shape)
print("conditions shape:", conditions.shape)
print("interventions shape:", interventions.shape)

Saved files:
- /content/cleaned_outputs/clinical_trials_cleaned.csv
- /content/cleaned_outputs/clinical_trials.csv
- /content/cleaned_outputs/organizations.csv
- /content/cleaned_outputs/conditions.csv
- /content/cleaned_outputs/interventions.csv
clinical_trials shape: (496615, 10)
organizations shape: (38106, 4)
conditions shape: (916564, 3)
interventions shape: (901498, 4)
